# Reproduce MRL notebook analyses

Execute the topic notebooks with a resource-aware schedule. Figure ownership is
declared inside each notebook; there is no external publication-manifest dependency.
Publication mode also executes the corrected predictive-linkage notebook and fails
closed unless all required Au and ITO raw files are available.

In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import os, subprocess, sys

RUN_PROFILE = os.getenv("MRL_RUN_PROFILE", "reduced")  # reduced | publication | smoke
DEVICE = os.getenv("MRL_DEVICE", "auto")
WORKERS = os.getenv("MRL_WORKERS", "auto")
_publication_default = "1" if RUN_PROFILE == "publication" else "0"
SAVE_FIGURES = os.getenv("MRL_SAVE_FIGURES", _publication_default) == "1"
SAVE_RESULTS = os.getenv("MRL_SAVE_RESULTS", _publication_default) == "1"
OUTPUT_DIR = Path(os.getenv("MRL_OUTPUT_DIR", "generated_figures"))
OVERWRITE = os.getenv("MRL_OVERWRITE", "0") == "1"
RUN_EXTERNAL_DATA = os.getenv("MRL_RUN_EXTERNAL_DATA", "0") == "1"
ALLOW_DATA_DOWNLOADS = os.getenv("MRL_ALLOW_DATA_DOWNLOADS", "0") == "1"
FAN_OUT_CPU = os.getenv("MRL_FAN_OUT", "1") == "1"
KERNEL_NAME = os.getenv("MRL_KERNEL_NAME", "python3")
EXECUTION_TIMEOUT = os.getenv(
    "MRL_NOTEBOOK_TIMEOUT", "-1" if RUN_PROFILE == "publication" else "7200"
)
if RUN_PROFILE == "publication" and not (SAVE_RESULTS and SAVE_FIGURES):
    raise RuntimeError("publication reproduction requires saved result tables and figures")

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "experiments" else HERE
EXP = ROOT / "experiments"
EXECUTED = EXP / ".executed" / RUN_PROFILE
TOPIC_NOTEBOOKS = [
    "00_device_physics_and_trace.ipynb",
    "01_distal_credit_ladder.ipynb",
    "02_sequential_and_scaling.ipynb",
    "03_deep_local_and_faults.ipynb",
    "04_biological_grounding.ipynb",
    "05_extensions.ipynb",
]
RUN_PREDICTIVE_LINKAGE = (RUN_PROFILE in {"reduced", "publication"} or
                          os.getenv("MRL_RUN_PREDICTIVE_LINKAGE", "0") == "1")
NOTEBOOKS = TOPIC_NOTEBOOKS + (["06_nmi_predictive_linkage.ipynb"] if RUN_PREDICTIVE_LINKAGE else [])
OUTER_FAN = TOPIC_NOTEBOOKS[0:2] + TOPIC_NOTEBOOKS[4:5]
INNER_PARALLEL = TOPIC_NOTEBOOKS[2:4] + TOPIC_NOTEBOOKS[5:6]
if RUN_PREDICTIVE_LINKAGE:
    INNER_PARALLEL.append("06_nmi_predictive_linkage.ipynb")

if WORKERS == "auto":
    RESOLVED_WORKERS = max(1, min(6, (os.cpu_count() or 4) - 2))
else:
    RESOLVED_WORKERS = max(1, int(WORKERS))


def execute_notebook(name, *, inner_workers):
    env = os.environ.copy()
    env.update({
        "MRL_RUN_PROFILE": RUN_PROFILE,
        "MRL_DEVICE": DEVICE,
        "MRL_WORKERS": str(inner_workers),
        "MRL_SAVE_FIGURES": "1" if SAVE_FIGURES else "0",
        "MRL_SAVE_RESULTS": "1" if SAVE_RESULTS else "0",
        "MRL_OUTPUT_DIR": str(OUTPUT_DIR),
        "MRL_OVERWRITE": "1" if OVERWRITE else "0",
        "MRL_RUN_EXTERNAL_DATA": "1" if RUN_EXTERNAL_DATA else "0",
        "MRL_ALLOW_DATA_DOWNLOADS": "1" if ALLOW_DATA_DOWNLOADS else "0",
    })
    if inner_workers == 1:
        env["MRL_CHILD_PROCESS"] = "1"
    else:
        env.pop("MRL_CHILD_PROCESS", None)
    EXECUTED.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "-m", "nbconvert", "--to", "notebook", "--execute",
        f"--ExecutePreprocessor.kernel_name={KERNEL_NAME}",
        f"--ExecutePreprocessor.timeout={EXECUTION_TIMEOUT}",
        "--output", name, "--output-dir", str(EXECUTED.resolve()),
        str((EXP / name).resolve()),
    ]
    done = subprocess.run(command, cwd=ROOT, env=env, text=True, capture_output=True)
    if done.returncode:
        raise RuntimeError(f"{name} failed:\n{done.stdout}\n{done.stderr}")
    return name


completed = []
if os.getenv("MRL_SKIP_EXECUTION", "0") != "1":
    if FAN_OUT_CPU:
        with ThreadPoolExecutor(max_workers=len(OUTER_FAN)) as pool:
            futures = {pool.submit(execute_notebook, name, inner_workers=1): name
                       for name in OUTER_FAN}
            for future in as_completed(futures):
                future.result()
        completed.extend(OUTER_FAN)
    else:
        # With no outer pool, give each notebook the full worker budget.
        completed.extend(execute_notebook(name, inner_workers=RESOLVED_WORKERS)
                         for name in OUTER_FAN)

    # Each heavy notebook owns the whole worker budget in turn; no nested pools.
    completed.extend(execute_notebook(name, inner_workers=RESOLVED_WORKERS)
                     for name in INNER_PARALLEL)

completed = [name for name in NOTEBOOKS if name in completed]
print({"profile": RUN_PROFILE, "executed": completed, "outer_fan": FAN_OUT_CPU,
       "inner_workers": RESOLVED_WORKERS, "figures_saved": SAVE_FIGURES,
       "results_saved": SAVE_RESULTS, "kernel": KERNEL_NAME,
       "timeout_s": EXECUTION_TIMEOUT})

{'profile': 'reduced', 'executed': ['00_device_physics_and_trace.ipynb', '01_distal_credit_ladder.ipynb', '02_sequential_and_scaling.ipynb', '03_deep_local_and_faults.ipynb', '04_biological_grounding.ipynb', '05_extensions.ipynb', '06_nmi_predictive_linkage.ipynb'], 'outer_fan': False, 'inner_workers': 4, 'figures_saved': False, 'results_saved': False, 'kernel': 'python3', 'timeout_s': '7200'}


## Interpretation

All numerical panels are generated from measured fixtures or live model
calls. Full numerical archives are used only when a topic notebook's explicit
archive opt-in is enabled. DANDI 001340 logged replay remains absent until its pinned
recordings and validated state-free caches are enabled; EEG is descriptive only and
never gates learning. The driver never substitutes toy data or a reference raster.
Saving remains opt-in for smoke/reduced runs and is mandatory in publication mode.